In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_chroma import Chroma
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.memory import ConversationBufferMemory

a:\Programming-Language-\LangGraph&Langchain\USING_LANGCHAIN_PROJECT\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain.chains'

In [2]:
PDF_PATH = "book.pdf"

In [3]:

# Directory to persist the Chroma database locally
CHROMA_PERSIST_DIR = "./chroma_db"

In [4]:
def initialize_vector_db():
    """Loads a PDF, chunks the text, and stores embeddings in ChromaDB."""
    print("Loading and processing PDF...")
    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()

    # Split text into manageable chunks for the model context window
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_documents(documents)

    # Use a fast, local open-source embedding model
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Create and persist the Chroma vector database
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_PERSIST_DIR
    )
    print("Vector database initialized and persisted.")
    return vector_db


In [ ]:



def build_chatbot(vector_db):
    """Constructs the Conversational Chain with memory and HF LLM."""
    
    # Define the open-source LLM (Mistral 7B Instruct is highly capable)
    llm = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.2",
        temperature=0.1,
        max_new_tokens=512
    )

    # Initialize memory to track the chat history ("hai feature")
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Create the retrieval chain
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}), # Retrieve top 3 chunks
        memory=memory
    )
    return qa_chain

if __name__ == "__main__":
    # 1. Process PDF and Setup DB (Run this once or when the PDF changes)
    if not os.path.exists(CHROMA_PERSIST_DIR):
        vector_db = initialize_vector_db()
    else:
        # Load existing DB
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vector_db = Chroma(persist_directory=CHROMA_PERSIST_DIR, embedding_function=embeddings)
        print("Loaded existing vector database.")

    # 2. Build the Chatbot
    chat_chain = build_chatbot(vector_db)

    # 3. Interactive Loop in Terminal
    print("\nChatbot initialized. Type 'exit' to quit.")
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() == 'exit':
            break
            
        # Execute the chain
        response = chat_chain.invoke({"question": user_input})
        print(f"\nBot: {response['answer']}")